# Test Each Part of The Inference Pipeline

# DISTILLBERT

In [2]:
import os
import json
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer
import sys

# Add parent directory to sys.path so we can import from the project root (useful for Jupyter or script)
# os.getcwd()        → returns current working directory, e.g., "/path/to/symptom-ner/v01"
# os.path.join(..., "..") → moves one directory up, i.e., "/path/to/symptom-ner"
# os.path.abspath()  → resolves this to the absolute path
PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)

from gcp_utils import download_from_gcs, list_bucket_files
from config import settings


# ------- Load labels and test data - LOCALLY -------
with open("data/distillbert_splits/test.jsonl", "r") as f:
    test_data = []
    for line in f:
        test_data.append(line)

with open("data/id2label.json", "r") as f:
    id2label = json.load(f)
with open("data/label2id.json", "r") as f:
    label2id = json.load(f)
# ------------------------------------------------------

# Convert id2label keys from strings to integers (JSON loads keys as strings)
if any(isinstance(k, str) for k in id2label.keys()):
    id2label = {int(k): v for k, v in id2label.items()}


# CONFIG FOR LOADING FROM GCS 

# v01/runs/distilbert-base-uncased/run_0/
VERSION = "v01"
MODEL_NAME = "distilbert-base-uncased"  # or "dmis-lab/biobert-base-cased-v1.2" for BioBERT
RUN_IDX = 0  

GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{RUN_IDX}"
BUCKET_NAME = settings.BUCKET_NAME  # "ner_training_data_results"


/Users/robertagarcia/Desktop/learning/bert_symptom_ner/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Create a local directory path for the model
LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{RUN_IDX}"

# Check if model already exists locally
if os.path.exists(LOCAL_MODEL_DIR) and os.path.isfile(os.path.join(LOCAL_MODEL_DIR, "config.json")):
    print(f"✅ Model found locally at {LOCAL_MODEL_DIR}")
    print("Skipping download from GCS.")
else:
    # Download the model directory from GCS if not found locally
    print(f"📥 Model not found locally. Downloading from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
    downloaded_path = download_from_gcs(
        gcs_path=GCS_MODEL_PATH,
        local_path=LOCAL_MODEL_DIR,
        bucket_name=BUCKET_NAME
    )
    if downloaded_path:
        print(f"✅ Download complete. Model saved to {LOCAL_MODEL_DIR}")

# Load the model
print(f"📂 Loading model from {LOCAL_MODEL_DIR}...")
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)

# Move to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
print(f"Using device: {device}")
model.to(device)
model.eval()

✅ Model found locally at ./downloaded_models/distilbert-base-uncased/run_0
Skipping download from GCS.
📂 Loading model from ./downloaded_models/distilbert-base-uncased/run_0...
Using device: mps


DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
   

# **Token Level Prediction**

In [4]:
from inference_utils import predict_token_level
test_text = "Patient reports severe headache and nausea"
tokens, predictions = predict_token_level(test_text, model, tokenizer, device=device)
print(f"Tokens {len(tokens)}:\n\t{tokens}")
print(f"Predictions {len(predictions)}:\n\t{predictions}")

Tokens 6:
	['patient', 'reports', 'severe', 'headache', 'and', 'nausea']
Predictions 6:
	[4, 4, 1, 3, 3, 3]


In [4]:
# RUN ANOTHER EXAMPLE: 
sample = json.loads(test_data[1])
text = sample.get('text')
tokens = sample.get('tokens')
token_label_ids = sample.get('token_label_ids')
print(f"TEXT: {text}")
print(f"TOKENS from test data: {tokens}")
tks, predictions = predict_token_level(text, model, tokenizer, device=device)
print(f"Returned tokens: {tks}")
print(f"Predictions: {predictions[0]}")


TEXT: The patient has lymphatic system symptom.
TOKENS from test data: ['[CLS]', 'the', 'patient', 'has', 'l', '##ym', '##pha', '##tic', 'system', 'sy', '##mpt', '##om', '.', '[SEP]']
Returned tokens: ['the', 'patient', 'has', 'l', '##ym', '##pha', '##tic', 'system', 'sy', '##mpt', '##om', '.']
Predictions: 4


# **Word Level Prediction**

In [5]:
from inference_utils import predict_word_level

sample = json.loads(test_data[1])
text = sample.get('text')

predict_word_level(text=text, model=model, tokenizer=tokenizer, id2label=id2label, device=device)

(['the',
  'patient',
  'has',
  'l',
  '##ym',
  '##pha',
  '##tic',
  'system',
  'sy',
  '##mpt',
  '##om',
  '.'],
 ['O',
  'O',
  'O',
  'B-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'O'],
 [0, 1, 2, 3, 3, 3, 3, 4, 5, 5, 5, 6],
 ['The', 'patient', 'has', 'lymphatic', 'system', 'symptom', '.'],
 ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O'])

In [6]:
from inference_utils import predict_word_level

#sample = json.loads(test_data[1])
text = "The patient has cataplexy." #lymphatic system symptom and cataplexy."#sample.get('text')
samples = ["The patient has cataplexy.", "The patient has lymphatic system symptom.", "The patient has lymphatic system symptom and cataplexy.","The patient has lymphatic system symptom.", "The patient has cataplexy and lymphatic system symptom.", "Roberta does not have back pain but she has an inflamation in her wrist"]
for s in samples:
    print("="*20)
    print(f" TEXT: {s}")
    print("="*20)
    tokens,token_labels,word_ids, words, word_labels =  predict_word_level(text=s, model=model, tokenizer=tokenizer, id2label=id2label, device=device)
    print("tokens: ", tokens)
    print("token labels: ", token_labels)
    print("word_ids: ", word_ids)
    print("word: ", words)
    print("bio word labels: ", word_labels)
    print()

 TEXT: The patient has cataplexy.
tokens:  ['the', 'patient', 'has', 'cat', '##ap', '##le', '##xy', '.']
token labels:  ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O']
word_ids:  [0, 1, 2, 3, 3, 3, 3, 4]
word:  ['The', 'patient', 'has', 'cataplexy', '.']
bio word labels:  ['O', 'O', 'O', 'B-SYMPTOM_POS', 'O']

 TEXT: The patient has lymphatic system symptom.
tokens:  ['the', 'patient', 'has', 'l', '##ym', '##pha', '##tic', 'system', 'sy', '##mpt', '##om', '.']
token labels:  ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O']
word_ids:  [0, 1, 2, 3, 3, 3, 3, 4, 5, 5, 5, 6]
word:  ['The', 'patient', 'has', 'lymphatic', 'system', 'symptom', '.']
bio word labels:  ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O']

 TEXT: The patient has lymphatic system symptom and cataplexy.
tokens:  ['the', 'patient', 'has', 'l', '##ym', 

# Test word_labels --> entity_spans

In [6]:
examples = {
    "ex1" : [['The', 'patient', 'has', 'lymphatic', 'system', 'symptom', '.'],
    ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O']
    ],
    "ex2" : [
        ['The', 'patient', 'has', 'cataplexy', '.'],
        ['O', 'O', 'O', 'B-SYMPTOM_POS', 'O']    
    ],
    "ex3" : [
        ['Roberta', 'does', 'not', 'have', 'back', 'pain', 'but', 'she', 'has', 'an', 'inflamation', 'in', 'her', 'wrist'],
        ['O', 'O', 'O', 'O', 'B-SYMPTOM_NEG', 'I-SYMPTOM_NEG', 'O', 'O', 'O', 'O', 'I-SYMPTOM_POS', 'I-SYMPTOM_NEG', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS']
    ]
  
}


In [ ]:
from inference_utils import word_labels_to_spans

word_labels_to_spans(examples["ex1"][0], examples["ex1"][1])

[{'start': 0, 'end': 3, 'text': 'The', 'label': 'O'},
 {'start': 4, 'end': 11, 'text': 'patient', 'label': 'O'},
 {'start': 12, 'end': 15, 'text': 'has', 'label': 'O'},
 {'start': 16,
  'end': 40,
  'text': 'lymphatic system symptom',
  'label': 'SYMPTOM_POS'},
 {'start': 41, 'end': 42, 'text': '.', 'label': 'O'}]

In [8]:
word_labels_to_spans(examples["ex2"][0], examples["ex2"][1])

[{'start': 0, 'end': 3, 'text': 'The', 'label': 'O'},
 {'start': 4, 'end': 11, 'text': 'patient', 'label': 'O'},
 {'start': 12, 'end': 15, 'text': 'has', 'label': 'O'},
 {'start': 16, 'end': 25, 'text': 'cataplexy', 'label': 'SYMPTOM_POS'},
 {'start': 26, 'end': 27, 'text': '.', 'label': 'O'}]

In [9]:
word_labels_to_spans(examples["ex3"][0], examples["ex3"][1])

[{'start': 0, 'end': 7, 'text': 'Roberta', 'label': 'O'},
 {'start': 8, 'end': 12, 'text': 'does', 'label': 'O'},
 {'start': 13, 'end': 16, 'text': 'not', 'label': 'O'},
 {'start': 17, 'end': 21, 'text': 'have', 'label': 'O'},
 {'start': 22, 'end': 31, 'text': 'back pain', 'label': 'SYMPTOM_NEG'},
 {'start': 32, 'end': 35, 'text': 'but', 'label': 'O'},
 {'start': 36, 'end': 39, 'text': 'she', 'label': 'O'},
 {'start': 40, 'end': 43, 'text': 'has', 'label': 'O'},
 {'start': 44, 'end': 46, 'text': 'an', 'label': 'O'},
 {'start': 47, 'end': 58, 'text': 'inflamation', 'label': 'SYMPTOM_POS'},
 {'start': 59, 'end': 61, 'text': 'in', 'label': 'SYMPTOM_NEG'},
 {'start': 62, 'end': 71, 'text': 'her wrist', 'label': 'SYMPTOM_POS'}]

# Testing the Real World Cases with the BEST MODEL: BioBERT run_2!


In [1]:
import os, json, torch, sys
from transformers import AutoModelForTokenClassification, AutoTokenizer


PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)

# Local imports 
from gcp_utils import download_from_gcs
from config import settings
from inference_utils import word_labels_to_spans, predict_word_level

# ------- Load labels and test data - LOCALLY -------
with open("data/biobert_splits/test.jsonl", "r") as f:
    test_data = []
    for line in f:
        test_data.append(line)

with open("data/id2label.json", "r") as f:
    id2label = json.load(f)
with open("data/label2id.json", "r") as f:
    label2id = json.load(f)
# ------------------------------------------------------

# Convert id2label keys from strings to integers (JSON loads keys as strings)
if any(isinstance(k, str) for k in id2label.keys()):
    id2label = {int(k): v for k, v in id2label.items()}


# CONFIG FOR LOADING FROM GCS 

# v01/runs/distilbert-base-uncased/run_0/
VERSION = "v01"
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1" 
RUN_IDX = 2

GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{RUN_IDX}"
BUCKET_NAME = settings.BUCKET_NAME  # "ner_training_data_results"

# Create a local directory path for the model
LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{RUN_IDX}"

# Check if model already exists locally
if os.path.exists(LOCAL_MODEL_DIR) and os.path.isfile(os.path.join(LOCAL_MODEL_DIR, "config.json")):
    print(f"✅ Model found locally at {LOCAL_MODEL_DIR}")
    print("Skipping download from GCS.")
else:
    # Download the model directory from GCS if not found locally
    print(f"📥 Model not found locally. Downloading from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
    downloaded_path = download_from_gcs(
        gcs_path=GCS_MODEL_PATH,
        local_path=LOCAL_MODEL_DIR,
        bucket_name=BUCKET_NAME
    )
    if downloaded_path:
        print(f"✅ Download complete. Model saved to {LOCAL_MODEL_DIR}")

# Load the model
print(f"📂 Loading model from {LOCAL_MODEL_DIR}...")
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)

# Move to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
print(f"Using device: {device}")
model.to(device)
model.eval()

/Users/robertagarcia/Desktop/learning/bert_symptom_ner/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Model found locally at ./downloaded_models/dmis-lab/biobert-base-cased-v1.1/run_2
Skipping download from GCS.
📂 Loading model from ./downloaded_models/dmis-lab/biobert-base-cased-v1.1/run_2...
Using device: mps


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [4]:
# Import gold-standard cases
from real_world_cases import REAL_WORLD_CASES

for case in REAL_WORLD_CASES:
    text = case["text"]
    expected_entities = case.get("expected_entities", [])
    print(f"\n=== Case: {case['id']} ===")
    print(f"Text: {text}")

    tokens, token_labels, word_ids, words, word_labels = predict_word_level(
        text=text,
        model=model,
        tokenizer=tokenizer,
        id2label=id2label,
        device=device,
    )
    spans = word_labels_to_spans(words, word_labels)
  
    spans = [span_data for span_data in spans if span_data['label'] != 'O']
    print("Tokens:", tokens)
    print("Token-level labels:", token_labels)
    print("Word-level label indices:", word_labels)
    print("Predicted spans (Excluding 'O'):", spans)
    print("Expected entities:", expected_entities)



=== Case: case_simple_single ===
Text: The patient has a large blister on her toe.
Tokens: ['the', 'patient', 'has', 'a', 'large', 'b', '##list', '##er', 'on', 'her', 'toe', '.']
Token-level labels: ['O', 'O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O', 'I-SYMPTOM_POS', 'O']
Word-level label indices: ['O', 'O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O', 'I-SYMPTOM_POS', 'O']
Predicted spans (Excluding 'O'): [{'start': 18, 'end': 34, 'text': 'large blister on', 'label': 'SYMPTOM_POS'}, {'start': 39, 'end': 42, 'text': 'toe', 'label': 'SYMPTOM_POS'}]
Expected entities: [{'text': 'blister', 'label': 'SYMPTOM_POS'}]

=== Case: case_multiword_single ===
Text: The patient has lymphatic system symptom.
Tokens: ['the', 'patient', 'has', 'l', '##ymph', '##atic', 'system', 's', '##ym', '##pt', '##om', '.']
Token-level labels: ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPT